In [11]:
import json
from sentence_transformers import util
import numpy as np
from scipy import stats
import os
import pandas as pd


# Analysis code

In [9]:
def load_embeddings_from_jsonl(file_path):
    """
    Loads embeddings from a .jsonl file.

    Args:
        file_path (str): The path to the .jsonl file.

    Returns:
        np.array: A numpy array of sentence embeddings.
    """
    embeddings = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            embeddings.append(item['embedding'])
    return np.array(embeddings)


def calculate_semantic_metrics(embeddings):
    """
    Calculates semantic diversity and related metrics from sentence embeddings.

    Args:
        embeddings (np.array): A numpy array of sentence embeddings.

    Returns:
        tuple: A tuple containing:
            - cosine_distances (np.array): Pairwise cosine distances.
            - sem_div (float): The semantic diversity score.
            - std_err (float): The standard error of the cosine distances.
    """
    similarities = util.pytorch_cos_sim(embeddings, embeddings).numpy()
    cosine_distances = 1 - similarities

    # Exclude self-comparisons from the diagonal
    mask = np.ones_like(cosine_distances, dtype=bool)
    np.fill_diagonal(mask, False)
    pairwise_distances = cosine_distances[mask]

    sem_div = np.mean(pairwise_distances)
    # ddof=1 for sample standard deviation
    std_dev = np.std(pairwise_distances, ddof=1)
    std_err = std_dev / np.sqrt(pairwise_distances.size)

    return cosine_distances, sem_div, std_err


def process_embeddings_directory(input_dir, output_dir):
    """
    Processes all embedding files in a directory, calculates semantic metrics,
    and saves the results.

    Args:
        input_dir (str): The directory containing .jsonl embedding files.
        output_dir (str): The directory where results will be saved.

    Returns:
        pd.DataFrame: A DataFrame containing the summary of results.
                      Returns None if no files were processed.
    """
    os.makedirs(output_dir, exist_ok=True)
    results = []

    jsonl_files = [f for f in os.listdir(input_dir) if f.endswith('.jsonl')]
    
    if not jsonl_files:
        print(f"No .jsonl files found in '{input_dir}'.")
        return None

    for filename in jsonl_files:
        print(f"Processing {filename}...")
        input_path = os.path.join(input_dir, filename)

        try:
            embeddings = load_embeddings_from_jsonl(input_path)
        except (json.JSONDecodeError, IOError) as e:
            print(f"  Error reading {filename}: {e}. Skipping.")
            continue
            
        if embeddings.size == 0 or embeddings.ndim != 2:
            print(f"  Warning: No valid embeddings found in {filename}. Skipping.")
            continue

        cosine_distances, sem_div, std_err = calculate_semantic_metrics(embeddings)

        base_name = os.path.splitext(filename)[0]
        distances_filename = f"cosine_distances_{base_name}.npy"
        distances_output_path = os.path.join(output_dir, distances_filename)
        np.save(distances_output_path, cosine_distances)

        results.append({
            'filename': base_name,
            'semantic_diversity': sem_div,
            'standard_error': std_err
        })

        print(f"  Semantic Diversity: {sem_div:.4f} ± {std_err:.6f}")
        print(f"  Saved cosine distances to '{distances_output_path}'")

    if not results:
        print("\nNo files were successfully processed.")
        return None

    results_df = pd.DataFrame(results)
    csv_output_path = os.path.join(output_dir, 'semantic_diversity_summary.csv')
    results_df.to_csv(csv_output_path, index=False)

    print(f"\nSummary of results saved to '{csv_output_path}'")
    return results_df

def get_label_from_filename(base_name):
    """Creates a readable label from a filename."""
    if 'single' in base_name:
        return 'Single-Source-Tuned'
    if 'multi' in base_name:
        return 'Multi-Source-Tuned'
    if 'human' in base_name:
        return 'Human-Tuned'
    if 'dolly_test' in base_name:
        return 'Vanilla (Llama)'
    return base_name # Fallback to the original name if no keyword is found

def perform_statistical_analysis(output_dir, summary_df):
    """
    Performs Levene's test and Kruskal-Wallis analysis on the generated
    cosine distance files.
    """
    all_distances = {}
    
    # Load all the cosine distance arrays
    for _, row in summary_df.iterrows():
        base_name = row['filename']
        label = get_label_from_filename(base_name)
        distances_path = os.path.join(output_dir, f"cosine_distances_{base_name}.npy")
        
        if os.path.exists(distances_path):
            cos_dist_matrix = np.load(distances_path)
            mask = np.ones_like(cos_dist_matrix, dtype=bool)
            np.fill_diagonal(mask, False)
            all_distances[label] = cos_dist_matrix[mask]
        else:
            print(f"Warning: Could not find distances file: {distances_path}")

    if len(all_distances) < 2:
        print("Need at least two groups to perform statistical tests.")
        return

    labels = list(all_distances.keys())
    groups = list(all_distances.values())

    print("\n--- Statistical Analysis ---")

    # For very large datasets, tests can become overly sensitive.
    # We sample the data to get more meaningful results, as in the original notebook.
    np.random.seed(42)
    n_sample = 10000
    
    can_sample = all(len(g) >= n_sample for g in groups)
    
    if can_sample:
        print(f"\nSampling {n_sample} distances from each group for statistical tests.")
        sampled_groups = [np.random.choice(g, size=n_sample, replace=False) for g in groups]
    else:
        print(f"\nOne or more groups has fewer than {n_sample} elements. Using full datasets for tests.")
        sampled_groups = groups

    # 1. Levene's Test for Homogeneity of Variances
    print("\n1. Levene's Test for Homogeneity of Variances:")
    stat_levene, p_levene = stats.levene(*sampled_groups)
    print(f"  Levene's test statistic: {stat_levene:.4f}, p-value: {p_levene:.4f}")
    if p_levene < 0.05:
        print("  Result: Variances are significantly different (p < 0.05).")
    else:
        print("  Result: Variances are not significantly different (p >= 0.05).")

    # 2. Kruskal-Wallis H-Test
    print("\n2. Kruskal-Wallis H-Test (non-parametric ANOVA):")
    H, p_kruskal = stats.kruskal(*sampled_groups)
    print(f"  H-statistic: {H:.2f}, p-value: {p_kruskal:.4f}")

    # 3. Pairwise Comparisons (if Kruskal-Wallis is significant)
    if p_kruskal < 0.05:
        print("\n3. Pairwise Comparisons (Mann-Whitney U test with Bonferroni correction):")
        
        n_comparisons = len(labels) * (len(labels) - 1) // 2
        alpha = 0.05 / n_comparisons
        print(f"  Corrected significance level (alpha): {alpha:.6f}")

        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                stat_mw, p_mw = stats.mannwhitneyu(sampled_groups[i], sampled_groups[j], alternative='two-sided')
                print(f"\n  - {labels[i]} vs {labels[j]}:")
                print(f"    U-statistic: {stat_mw:.2f}, p-value: {p_mw:.2e}")
                print(f"    Significant difference: {'Yes' if p_mw < alpha else 'No'}")
    else:
        print("\nKruskal-Wallis test is not significant; no pairwise comparisons needed.")


In [12]:
INPUT_DIR = "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/sentence_embeddings"
parent_dir = os.path.dirname(INPUT_DIR)
OUTPUT_DIR = os.path.join(parent_dir, "semantic_diversity_results")

# 2. Run the main processing pipeline to calculate metrics
summary_df = process_embeddings_directory(INPUT_DIR, OUTPUT_DIR)

# 3. If processing was successful, run the statistical analysis
if summary_df is not None:
    perform_statistical_analysis(OUTPUT_DIR, summary_df)
    
    print("\n--- Results Summary ---")
    display(summary_df) # Use display() for better formatting in notebooks


Processing dolly_test_Llama_embeddings.jsonl...
  Semantic Diversity: 0.9547 ± 0.000018
  Saved cosine distances to '/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/semantic_diversity_results/cosine_distances_dolly_test_Llama_embeddings.npy'
Processing output_lora_llama_8b_multi_embeddings.jsonl...
  Semantic Diversity: 0.9535 ± 0.000018
  Saved cosine distances to '/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/semantic_diversity_results/cosine_distances_output_lora_llama_8b_multi_embeddings.npy'
Processing output_lora_llama_8b_single_embeddings.jsonl...
  Semantic Diversity: 0.9565 ± 0.000017
  Saved cosine distances to '/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/semantic_diversity_results/cosine_distances_output_lora_llama_8b_single_embeddings.npy'
Processing output_lora_llama_8b_human_embeddings.jsonl...
  Semantic Diversity: 0.9680 ± 0.000018
  Saved cosine distances to '/Users/maxschaffelder/Desktop/Thesis/data/exp_1/small/sem_div/s

,filename,semantic_diversity,standard_error
0,dolly_test_Llama_embeddings,0.954650,0.000018
1,output_lora_llama_8b_multi_embeddings,0.953474,0.000018
2,output_lora_llama_8b_single_embeddings,0.956516,0.000017
3,output_lora_llama_8b_human_embeddings,0.968035,0.000018
